# 🎙️ Hindi–English Video & Audio Transcription

**Model:** Whisper `medium` via `faster-whisper` (CTranslate2)  
**Speed:** ~3–4× faster than `large-v3`, excellent for Hindi/English code-switching  
**Cost:** Free — runs on Colab T4 GPU or CPU  

### Before you start
1. **Runtime → Change runtime type → T4 GPU** (strongly recommended)  
2. Run all cells top to bottom  
3. Upload your video/audio + optional `.vtt` reference when prompted  

### What this notebook does
- Transcribes Hindi-dominant audio (with English) using Whisper Medium  
- Optionally maps speaker names from an existing `.vtt` file (timing-based, not diarization)  
- Outputs: `.txt` transcript, `.vtt` subtitles, `.json` with word-level timestamps  
- Downloads everything as a single `.zip`  

> ⚠️ **No ASR is perfect.** Review names, numbers, acronyms, and overlapping speech manually.

---
## 1 · Install dependencies

In [ ]:
%%time
# Install ffmpeg (needed for audio extraction)
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null 2>&1

# Install CUDA runtime libs needed by ctranslate2 on Colab
!pip -q install nvidia-cublas-cu12 nvidia-cudnn-cu12

# Pin av<18 — PyAV 18+ removed 'metadata_errors' kwarg that faster-whisper uses
!pip -q install "av<18"

# Install faster-whisper from GitHub (latest dev — has the PyAV 18+ fix)
!pip -q install --force-reinstall --no-deps git+https://github.com/SYSTRAN/faster-whisper.git

print("✅ Dependencies installed.")

---
## 2 · Upload your files

Upload a **video** (`.mp4`, `.mkv`, `.mov`, `.avi`, `.webm`) or **audio** (`.wav`, `.mp3`, `.m4a`, `.flac`, `.ogg`, `.aac`).  
Optionally upload a **`.vtt`** reference file for speaker-name hints.  

If files are already in `/content`, this cell finds them automatically.

In [ ]:
from google.colab import files
from pathlib import Path

MEDIA_EXTENSIONS = {
    ".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v",
    ".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac",
}

content_files = [p for p in Path("/content").iterdir() if p.is_file()]
media_files = [p for p in content_files if p.suffix.lower() in MEDIA_EXTENSIONS]
vtt_files = [p for p in content_files if p.suffix.lower() == ".vtt"]

if not media_files:
    print("📁 Upload your media file (and optional .vtt reference):")
    uploaded = files.upload()
    uploaded_paths = [Path("/content") / name for name in uploaded]
    media_files = [p for p in uploaded_paths if p.suffix.lower() in MEDIA_EXTENSIONS]
    vtt_files = [p for p in uploaded_paths if p.suffix.lower() == ".vtt"]

if not media_files:
    raise FileNotFoundError(
        "No media file found. Upload a video or audio file "
        "(.mp4, .mkv, .mov, .avi, .webm, .wav, .mp3, .m4a, .flac, .ogg, .aac)"
    )

MEDIA_PATH = str(media_files[0])
REFERENCE_VTT = str(vtt_files[0]) if vtt_files else None

print(f"\n✅ Media file:    {Path(MEDIA_PATH).name}")
print(f"{'✅' if REFERENCE_VTT else '⚠️'} Reference VTT: {Path(REFERENCE_VTT).name if REFERENCE_VTT else 'None (speaker labels will be Unknown)'}")
if len(media_files) > 1:
    print(f"\nℹ️  Multiple media files found: {[p.name for p in media_files]}")
    print(f"   Using: {Path(MEDIA_PATH).name}. Edit MEDIA_PATH above to change.")

---
## 3 · Configuration

| Setting | Value | Why |
|---------|-------|-----|
| `MODEL_SIZE` | `medium` | Best speed/accuracy balance for free Colab |
| `LANGUAGE` | `"hi"` | Prioritizes Hindi decoding while still catching English |
| `BEAM_SIZE` | `5` | Quality-oriented beam search |
| `VAD_FILTER` | `True` | Skips silence → faster transcription |

In [ ]:
# ─── Transcription settings ──────────────────────────────────────
MODEL_SIZE  = "medium"       # Options: tiny, base, small, medium, large-v3
LANGUAGE    = "hi"           # "hi" for Hindi-dominant. Set to None for auto-detect.
BEAM_SIZE   = 5              # Higher = more accurate but slower (1–10)
VAD_FILTER  = True           # Skip silence regions for speed
# ─────────────────────────────────────────────────────────────────

OUTPUT_DIR = Path("/content/transcription_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Config ready: {MODEL_SIZE} model, language={LANGUAGE}, beam={BEAM_SIZE}, VAD={VAD_FILTER}")

---
## 4 · Load model & transcribe

- **GPU (T4):** Uses `float16` — fast and accurate  
- **CPU fallback:** Uses `int8` quantization — slower but works  

The `medium` model downloads ~1.5 GB on first run (cached after that).

> 💡 **Note:** `faster-whisper` uses **CTranslate2** (not PyTorch) — no `torch` needed!

In [ ]:
%%time
import time
import ctranslate2
from faster_whisper import WhisperModel

# ─── Device selection (via ctranslate2, NOT torch) ────────────────
def detect_device():
    """Detect best available device via ctranslate2."""
    try:
        supported = ctranslate2.get_supported_compute_types("cuda")
        if supported:
            print(f"🚀 CUDA available! Supported compute types: {supported}")
            # Prefer float16 if available, else auto
            if "float16" in supported:
                return "cuda", "float16"
            return "cuda", "auto"
    except Exception as e:
        print(f"ℹ️  CUDA check: {e}")
    return "cpu", "int8"

device, compute_type = detect_device()
if device == "cuda":
    print(f"🚀 Using GPU with {compute_type} precision")
else:
    print("⚠️  No GPU detected. Using CPU with int8 quantization (will be slower).")

# ─── Load model ─────────────────────────────────────────────────
print(f"\n📥 Loading Whisper {MODEL_SIZE} on {device} ({compute_type})...")
t0 = time.time()
model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)
print(f"   Model loaded in {time.time() - t0:.1f}s")

# ─── Transcribe ──────────────────────────────────────────────────
print(f"\n🎤 Transcribing: {Path(MEDIA_PATH).name}")
print(f"   This may take a few minutes depending on file length...\n")
t0 = time.time()

segments_iter, info = model.transcribe(
    MEDIA_PATH,
    language=LANGUAGE,
    task="transcribe",
    beam_size=BEAM_SIZE,
    vad_filter=VAD_FILTER,
    vad_parameters=dict(
        min_silence_duration_ms=500,   # Merge pauses shorter than 500ms
        speech_pad_ms=400,             # Pad speech boundaries
    ),
    word_timestamps=True,
    condition_on_previous_text=True,
    temperature=0.0,
)

# Collect segments (transcription is lazy — iteration does the work)
segments = []
for seg in segments_iter:
    segments.append({
        "start": float(seg.start),
        "end": float(seg.end),
        "text": seg.text.strip(),
        "avg_logprob": float(seg.avg_logprob),
        "no_speech_prob": float(seg.no_speech_prob),
        "words": [
            {
                "start": float(w.start),
                "end": float(w.end),
                "word": w.word,
                "probability": float(w.probability),
            }
            for w in (seg.words or [])
        ],
    })
    # Live progress
    mins = int(seg.end // 60)
    secs = int(seg.end % 60)
    print(f"  [{mins:02d}:{secs:02d}] {seg.text.strip()[:100]}")

elapsed = time.time() - t0

if not segments:
    raise RuntimeError(
        "❌ Whisper returned no segments. "
        "Check that the file contains audible speech and try again."
    )

# ─── Summary ─────────────────────────────────────────────────────
audio_duration = segments[-1]["end"]
speed_ratio = audio_duration / elapsed if elapsed > 0 else 0
print(f"\n{'─' * 50}")
print(f"✅ Transcription complete!")
print(f"   Language detected : {info.language} (confidence: {info.language_probability:.0%})")
print(f"   Segments          : {len(segments)}")
print(f"   Audio duration    : {audio_duration / 60:.1f} min")
print(f"   Processing time   : {elapsed / 60:.1f} min")
print(f"   Speed             : {speed_ratio:.1f}× realtime")

---
## 5 · Attach speaker labels from reference VTT

If you uploaded a `.vtt` file, this cell extracts speaker names and maps them to Whisper segments by **timestamp overlap** (≥50% overlap required).  

This is **not** speaker diarization — it simply borrows the speaker label from the reference cue whose timing best matches each Whisper segment. Without a `.vtt`, all speakers are labeled `Unknown`.

In [ ]:
import re

# ─── VTT parsing helpers ─────────────────────────────────────────
TIMESTAMP_RE = re.compile(r"(?:(\d+):)?(\d{2}):(\d{2})[.,](\d{3})")
SPEAKER_RE = re.compile(r"^([^:\n]{1,80}):\s*(.*)$", re.DOTALL)


def _ts_to_seconds(ts_str: str) -> float:
    """Convert VTT timestamp string to seconds."""
    m = TIMESTAMP_RE.fullmatch(ts_str.strip())
    if not m:
        raise ValueError(f"Invalid VTT timestamp: {ts_str!r}")
    h, mn, s, ms = m.groups()
    return int(h or 0) * 3600 + int(mn) * 60 + int(s) + int(ms) / 1000


def parse_vtt_speakers(path: str) -> list:
    """Extract speaker-labeled cues from a VTT file."""
    if not path:
        return []
    content = Path(path).read_text(encoding="utf-8-sig")
    cues = []
    for block in re.split(r"\n\s*\n", content):
        lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
        timing_idx = next((i for i, ln in enumerate(lines) if "-->" in ln), None)
        if timing_idx is None:
            continue
        parts = lines[timing_idx].split("-->", 1)
        start_m = TIMESTAMP_RE.search(parts[0])
        end_m = TIMESTAMP_RE.search(parts[1])
        if not start_m or not end_m:
            continue
        cue_text = re.sub(r"<[^>]*>", "", " ".join(lines[timing_idx + 1:])).strip()
        speaker_m = SPEAKER_RE.match(cue_text)
        if speaker_m:
            cues.append({
                "start": _ts_to_seconds(start_m.group()),
                "end": _ts_to_seconds(end_m.group()),
                "speaker": speaker_m.group(1).strip(),
            })
    return cues


# ─── Map speakers to segments ────────────────────────────────────
ref_cues = parse_vtt_speakers(REFERENCE_VTT)
labeled_segments = []

for seg in segments:
    duration = max(seg["end"] - seg["start"], 0.001)
    best_speaker, best_overlap = "Unknown", 0.0

    for cue in ref_cues:
        overlap = max(0.0, min(seg["end"], cue["end"]) - max(seg["start"], cue["start"]))
        if overlap > best_overlap:
            best_speaker, best_overlap = cue["speaker"], overlap

    if best_overlap / duration < 0.5:
        best_speaker = "Unknown"

    labeled_segments.append({**seg, "speaker": best_speaker})

# ─── Summary ─────────────────────────────────────────────────────
labeled_count = sum(1 for s in labeled_segments if s["speaker"] != "Unknown")
unique_speakers = sorted(set(s["speaker"] for s in labeled_segments if s["speaker"] != "Unknown"))

print(f"📋 Reference cues parsed : {len(ref_cues)}")
print(f"🏷️  Segments with speaker : {labeled_count} / {len(labeled_segments)}")
if unique_speakers:
    print(f"👥 Speakers found        : {', '.join(unique_speakers)}")
else:
    print("ℹ️  No speakers identified (no VTT reference or no speaker: prefix in cues)")

---
## 6 · Save & download results

Generates three output files and bundles them in a `.zip` for download:

| File | Contents |
|------|----------|
| `*_transcript.txt` | Human-readable speaker-labeled transcript |
| `*_transcript.vtt` | WebVTT subtitles (importable in video players) |
| `*_transcript.json` | Full data: word timestamps, confidence scores, metadata |

In [ ]:
import json
import zipfile


def fmt_ts(seconds: float) -> str:
    """Format seconds as HH:MM:SS.mmm for VTT."""
    ms = round(seconds * 1000)
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"


# ─── File names ──────────────────────────────────────────────────
stem = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(MEDIA_PATH).stem).strip("._") or "transcript"
txt_path  = OUTPUT_DIR / f"{stem}_transcript.txt"
vtt_path  = OUTPUT_DIR / f"{stem}_transcript.vtt"
json_path = OUTPUT_DIR / f"{stem}_transcript.json"

# ─── Write TXT ───────────────────────────────────────────────────
with txt_path.open("w", encoding="utf-8") as f:
    f.write(f"Transcript — {Path(MEDIA_PATH).name}\n")
    f.write(f"Model: Whisper {MODEL_SIZE} | Language: {info.language}\n")
    f.write("=" * 70 + "\n\n")
    for item in labeled_segments:
        ts = f"[{fmt_ts(item['start'])} --> {fmt_ts(item['end'])}]"
        f.write(f"{ts} {item['speaker']}: {item['text']}\n")

# ─── Write VTT ───────────────────────────────────────────────────
with vtt_path.open("w", encoding="utf-8") as f:
    f.write("WEBVTT\n\n")
    for i, item in enumerate(labeled_segments, 1):
        f.write(
            f"{i}\n"
            f"{fmt_ts(item['start'])} --> {fmt_ts(item['end'])}\n"
            f"{item['speaker']}: {item['text']}\n\n"
        )

# ─── Write JSON ──────────────────────────────────────────────────
output_meta = {
    "model": MODEL_SIZE,
    "language": info.language,
    "language_probability": round(info.language_probability, 4),
    "media_file": Path(MEDIA_PATH).name,
    "reference_vtt_used": bool(REFERENCE_VTT),
    "total_segments": len(labeled_segments),
    "audio_duration_seconds": round(segments[-1]["end"], 2),
    "segments": labeled_segments,
}
json_path.write_text(
    json.dumps(output_meta, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# ─── Bundle as ZIP ───────────────────────────────────────────────
zip_path = OUTPUT_DIR / f"{stem}_transcription.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in (txt_path, vtt_path, json_path):
        zf.write(p, arcname=p.name)

# ─── Summary & download ──────────────────────────────────────────
print("📦 Files saved:")
for p in (txt_path, vtt_path, json_path):
    size_kb = p.stat().st_size / 1024
    print(f"   {p.name} ({size_kb:.1f} KB)")
print(f"\n📥 Downloading {zip_path.name}...")
files.download(str(zip_path))

---
## 📝 Review checklist

After downloading, manually check:

- [ ] **Names & acronyms** — ASR often misspells proper nouns  
- [ ] **Hindi spellings** — Romanization may vary (e.g., "sahi" vs "sahih")  
- [ ] **Overlapping speech** — Segments where two people talk simultaneously  
- [ ] **Speaker labels** — Reference-based labels can be wrong if speakers switch mid-cue  
- [ ] **Low-confidence words** — Check the JSON `probability` field for words < 0.5  

### Tips
- If Hindi is under-recognized, try `LANGUAGE = None` for auto-detection  
- For higher accuracy (but slower), change `MODEL_SIZE = "large-v3"`  
- Reduce `BEAM_SIZE` to `1` for maximum speed (lower quality)